# Capítulo 7: Trabalhando com Dados

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 10 de Grus (2019).

> Especialistas costumam ter mais dados do que juízo.
>
> — Colin Powell

O [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) construiu a máquina que ajusta parâmetros. O [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) construiu os canos pelos quais o dado chega até um programa Python. Nenhum dos dois te prepara para o que acontece entre uma coisa e outra: o dado que chegou pelos canos do Capítulo 6 quase nunca está pronto para alimentar a máquina do Capítulo 5. Ele vem torto, incompleto, em unidades incompatíveis, com colunas demais ou de menos. Este capítulo é sobre essa distância. É também o capítulo em que as duas ferramentas do livro tomam os seus lugares. A [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) abre apresentando o array que a [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) mostrou de relance — daqui até o fim do livro é ele que faz as contas de cada modelo — e fecha mostrando a **fronteira**: a linha em que o `DataFrame` do [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html), onde o dado é lido, limpo e agrupado, entrega ao array uma matriz de números sem nome. Tudo o que este capítulo faz mora de um lado ou do outro dessa linha, e vale reparar em qual.

É um capítulo mais de artesanato do que de teoria. Não há um modelo novo para treinar nem uma prova para seguir — em vez disso, um punhado de técnicas pequenas e recorrentes: como olhar para um conjunto de dados antes de fazer qualquer coisa com ele, como representar uma linha de dados heterogênea sem reescrever a mesma lógica de conversão de tipo em cada função, como filtrar o que está sujo sem descartar o que só *parece* sujo, como agregar e comparar, como colocar dimensões em pé de igualdade antes de medir distância entre elas — e, por fim, como reduzir um conjunto de muitas dimensões a poucas, sem perder o que importa.

Ao final deste capítulo, você será capaz de:

- Explicar o que um array do `numpy` é — forma, tipo único, fatias, máscaras booleanas, *broadcasting* e reduções por eixo — e o que cada uma dessas operações substitui no código em listas dos capítulos [4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/index.html) e [5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html)
- Reconhecer a fronteira entre a mesa de trabalho e a calculadora — `df[colunas].to_numpy()` — e explicar por que ela é explícita
- Resumir e visualizar um conjunto de dados de uma, duas e muitas dimensões antes de tentar modelá-lo
- Representar uma linha de dados heterogênea com `NamedTuple`, e explicar por que isso resolve os dois problemas que um `dict` tem para essa tarefa
- Escrever o equivalente mutável com `dataclass`, reconhecer o compromisso que a mutabilidade reintroduz, e escolher entre registro tipado e `DataFrame` conforme a pergunta seja sobre **uma** linha ou sobre **todas**
- Escrever uma função de *parsing* que devolve `None` diante de dado ruim em vez de estourar, comparar essa política com a coerção silenciosa do `pandas`, e decidir o que fazer com as linhas rejeitadas
- Agrupar, ordenar e agregar dados tabulares com `groupby`, `sort_values` e `pct_change` — sabendo dizer que laço cada uma dessas linhas substitui —, e reconhecer onde o registro heterogêneo ainda pede um `NamedTuple`
- Reescalonar dados para que a unidade de cada dimensão pare de dominar o cálculo de distância
- Reduzir a dimensionalidade de um conjunto de dados com PCA, construído do zero, em arrays, sobre o mesmo `gradient_step` do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html)

## Seções

| Seção | Tópico |
|---|---|
| [7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) | Explorando Seus Dados |
| [7.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/02-namedtuples.html) | Usando NamedTuples |
| [7.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/03-dataclasses.html) | Dataclasses |
| [7.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-limpeza-e-transformacao.html) | Limpeza e Transformação |
| [7.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/05-manipulando-dados.html) | Manipulando Dados |
| [7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) | Reescalonamento |
| [7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-um-parenteses-tqdm.html) | Um Parêntese: tqdm |
| [7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html) | Redução de Dimensionalidade |

A última seção também cumpre uma promessa: o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html), ao discutir seleção de atributos, cita a redução de dimensionalidade "no Capítulo 7" como uma das formas de lidar com dados que têm atributos demais. É esta seção.

## Explorando Seus Dados

> **📌 Nota**
>
> Esta seção corresponde a *Exploring Your Data*, do capítulo 10 de Grus (2019).

Depois de identificar as perguntas que você quer responder e conseguir dado nas mãos — o assunto do [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) —, a tentação é pular direto para os modelos. Resista. O primeiro passo deveria ser sempre **explorar** os dados. Antes, porém, uma parada obrigatória. O [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) deixou o dado numa tabela do `pandas`, e é de lá que ele vem daqui para a frente; mas toda conta de modelo deste livro é feita em arrays do `numpy`. Este é o lugar de dizer o que um array é — e onde, exatamente, o dado deixa a tabela e vira array.

### De listas a arrays

O Capítulo 4 fechou cada seção com um callout que mostrava, de relance, o que o `numpy` faria no lugar do `dot` e do `subtract` de [vetores](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html) e do `get_column` de [matrizes](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/02-matrizes.html) que você escreveu — e a [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) usou um array pela primeira vez, lendo `dados/stocks.csv` com `np.loadtxt`. Este é o bloco em que o array deixa de ser relance: daqui até o fim do livro, todo vetor é um `np.ndarray`, e é ele a calculadora de cada modelo. Vale uma página para saber o que ele é.

Comece pelo `dot` do Capítulo 4, ao lado do que o `numpy` faz:

In [ ]:
import numpy as np

def dot(v, w):                       # o `dot` do Capítulo 4, sem alterações
    return sum(v_i * w_i for v_i, w_i in zip(v, w))

v = [1.0, 2.0, 3.0]
w = [4.0, 5.0, 6.0]

v_arr = np.array(v)
w_arr = np.array(w)

dot(v, w), v_arr @ w_arr

Os dois números são o mesmo 32 — a soma dos produtos, que você enxerga porque escreveu o `sum` sobre o `zip`. O que muda é o que está por baixo. Uma lista de `float` é um vetor de ponteiros para objetos Python, percorrido um a um pelo interpretador; um **`ndarray`** é um bloco contíguo de memória com números de um único tipo, e o `@` percorre esse bloco em código compilado. É por isso que o array existe: a conta é a mesma, o custo não — e o fim deste bloco mede a diferença.

#### Forma e tipo

Um array sabe duas coisas sobre si mesmo que uma lista não sabe. Tome as três pessoas da [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) — altura em polegadas, altura em centímetros, peso em libras —, uma por linha:

In [ ]:
X = np.array([[63, 160.0, 150],
              [67, 170.2, 160],
              [70, 177.8, 171]])

X.shape, X.dtype

`shape` é a forma: 3 linhas por 3 colunas — a matriz do Capítulo 4, só que o array a conhece sem que ninguém conte as listas. `dtype` é o tipo **único** de todos os elementos: bastou um `160.0` para tudo virar `float64`, inclusive o `63`. É a mesma regra que fez o `np.loadtxt` da seção 6.1 recusar uma coluna de texto — num array, ou tudo é número, ou nada é; um `str`, uma data e um `float` lado a lado não cabem nele, e é para esse caso que a [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/02-namedtuples.html) existe. Neste livro, uma matriz de dados `X` tem sempre uma linha por observação e uma coluna por atributo: `X.shape` é `(n, d)`.

#### Índices, fatias e colunas

A indexação começa como a de listas — `X[0]` é a primeira linha — e ganha um eixo a mais: `X[1, 2]` é a linha 1, coluna 2, e uma fatia pode correr **qualquer** eixo:

In [ ]:
X[0], X[1, 2], X[:, 0], X[1:, :2]

`X[:, 0]` é a coluna inteira das polegadas — o `get_column` do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/02-matrizes.html), sem percorrer linha nenhuma: o `:` significa "todas as linhas", e o `0` escolhe a coluna. `X[1:, :2]` recorta um bloco: da linha 1 em diante, as duas primeiras colunas.

#### Máscaras booleanas

Comparar um array com um número não devolve `True` ou `False`: devolve um array de booleanos, um por elemento. Esse array é uma **máscara**, e uma máscara serve de índice:

In [ ]:
pesados = X[:, 2] > 155        # quem pesa mais de 155 libras?
pesados, X[pesados], X[pesados, 0], pesados.sum()

`X[pesados]` fica só com as linhas em que a máscara é `True`; `X[pesados, 0]` combina a máscara com uma coluna. E como `True` conta como 1, `pesados.sum()` é quantos passaram no filtro. É o `if` de uma compreensão de lista, aplicado a todos os elementos de uma vez — e é assim que a [seção 7.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/05-manipulando-dados.html) vai separar "as linhas da AAPL" das outras 13,5 mil — do mesmo jeito, com a mesma sintaxe, sobre uma tabela em vez de sobre um array.

#### Broadcasting

Você já viu aritmética entre arrays na seção 6.1: `fechamento - abertura` subtraiu 23.105 pares de uma vez. O `numpy` estende a mesma regra a operandos de formas **diferentes**, e chama isso de *broadcasting*. O caso mais simples é um escalar contra um array — a coluna de polegadas vezes 2,54 devolve a de centímetros:

In [ ]:
X[:, 0] * 2.54, X[:, 1]

O caso que importa para o resto do livro é um vetor contra uma matriz. `X.mean(axis=0)` tem forma `(3,)` — uma média por coluna, explicada logo abaixo —, e subtraí-lo de `X`, que tem forma `(3, 3)`, subtrai o vetor **de cada linha**:

In [ ]:
X - X.mean(axis=0)

A regra: quando as formas diferem, o `numpy` alinha os eixos pela direita e estica o operando de tamanho 1 até casar com o outro. Um vetor `(3,)` contra uma matriz `(3, 3)` casa com as colunas, e cada linha recebe a subtração. É esta linha, `X - X.mean(axis=0)`, que a [seção 7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html) vai chamar de `de_mean`.

#### `axis`: em que direção reduzir

`X.mean()` sem argumento reduz o array inteiro a um número. Com `axis`, a redução corre um eixo só — e é aqui que a convenção "linha = observação, coluna = atributo" paga:

In [ ]:
X.mean(), X.mean(axis=0), X.mean(axis=1)

`axis=0` percorre as linhas e devolve um valor **por coluna** — a média de cada atributo, que é quase sempre o que se quer. `axis=1` percorre as colunas e devolve um valor por linha — a média das três medidas de uma pessoa, que aqui não significa nada, mas o `numpy` calcula do mesmo jeito. `sum`, `std`, `max` e `min` aceitam o mesmo `axis`.

#### O laço que some

Tudo acima cabia num `for`. O motivo de não escrever o `for` é o custo — e ele se mede. O mesmo `dot` do começo do bloco, sobre um milhão de elementos:

In [ ]:
import time

v_grande = np.linspace(0, 1, 1_000_000)
w_grande = np.linspace(1, 2, 1_000_000)
v_lista, w_lista = v_grande.tolist(), w_grande.tolist()

inicio = time.perf_counter()
soma_laco = dot(v_lista, w_lista)
tempo_laco = time.perf_counter() - inicio

inicio = time.perf_counter()
soma_array = v_grande @ w_grande
tempo_array = time.perf_counter() - inicio

assert abs(soma_laco - soma_array) < 1e-6
f"laço: {tempo_laco * 1000:.0f} ms; array: {tempo_array * 1000:.2f} ms; razão: {tempo_laco / tempo_array:.0f}x"

O resultado é o mesmo número; a diferença de tempo é de cerca de cem vezes — o número exato varia por máquina e por execução — e cresce com o tamanho. Não é o `for` que é lento — é cada iteração passar pelo interpretador, que precisa descobrir de novo, a cada `v_i * w_i`, que tipo de objeto está multiplicando. O `@` decide isso uma vez, para o bloco inteiro.

É a promessa que o callout do Capítulo 4 fez — *"depois desta seção, você olha `v @ w` e enxerga a soma dos produtos"* — cumprida; o que ele adiou de propósito foi usar o `@`. Você escreveu a soma; agora usa o `@`. Daqui em diante, cada laço sobre os pontos de um conjunto de dados vira uma dessas operações — fatia, máscara, *broadcasting*, redução por eixo —, e o laço que sobrevive é só o que **é** o algoritmo: os passos de um gradiente, as iterações de um agrupamento. Com isso na mão, dá para explorar os dados.

#### Do `DataFrame` para o array: a fronteira

Falta dizer de onde `X` vem. Quase nunca de uma matriz escrita à mão como a das três pessoas acima: o dado chega de um arquivo, e o [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) mostrou como — `pd.read_csv` devolve um **`DataFrame`**, a tabela de colunas nomeadas, cada uma com o seu tipo. É de lá que este livro parte daqui em diante.

Só que um modelo não sabe o que é uma coluna chamada `Close`. Ele recebe uma matriz de números e um vetor de números, e a passagem de um formato para o outro é um método:

In [ ]:
import pandas as pd

acoes = pd.read_csv("dados/stocks.csv", parse_dates=["Date"])

X = acoes[["Open", "High", "Low"]].to_numpy()   # a fronteira: a matriz do modelo
y = acoes["Close"].to_numpy()                   # e o alvo

X.shape, y.shape, X.dtype

Essas duas linhas são a **fronteira** deste livro, e vale dar nome a elas porque elas voltam em todo capítulo que ajusta um modelo. De um lado, a mesa de trabalho: o `DataFrame`, onde o dado é lido, tipado, limpo, juntado, agrupado e conferido — tudo com as colunas se chamando pelo nome. Do outro, a calculadora: `X` e `y`, dois blocos de `float64` sem nome nenhum, que é o que o `@`, o gradiente e a distância euclidiana sabem consumir. `.to_numpy()` é a passagem, e ela é explícita de propósito: **é aqui que o dado perde os nomes**, e é bom que isso apareça numa linha em vez de acontecer sozinho.

Repare que o nome `X` foi reaproveitado — a matriz das três pessoas cumpriu o papel dela. Daqui até o fim do livro, `X` é sempre a matriz `(n, d)` que o modelo recebe, e `y` o vetor `(n,)` do que se quer prever. Nada é ajustado aqui, e nem deveria: como exemplo de modelagem este par não presta — a máxima e a mínima do dia já contêm o fechamento, então prever `y` a partir desse `X` seria trapaça. O que importa aqui é a **forma**. O primeiro modelo ajustado sobre um `X` e um `y` como estes é o do [Capítulo 11](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html).

### Dados unidimensionais

O caso mais simples é um conjunto de dados unidimensional: só uma coleção de números — o tempo médio diário que cada usuário passa no seu site, por exemplo. Um primeiro passo óbvio é calcular estatísticas resumo: quantos pontos você tem, o menor, o maior, a média, o desvio padrão. Com um array são cinco chamadas — `len(x)`, `x.min()`, `x.max()`, `x.mean()`, `x.std(ddof=1)` (o `ddof=1` pede o desvio amostral; a [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) explica). Com uma tabela é uma só, e ela aparece mais adiante nesta seção.

Mas mesmo essas estatísticas não necessariamente dão uma boa noção do que está acontecendo. Um passo melhor é construir um **histograma**, agrupando os dados em faixas discretas — *buckets* — e contando quantos pontos caem em cada uma:

In [ ]:
from typing import Tuple

def bucketize(points: np.ndarray, bucket_size: float) -> np.ndarray:
    """Arredonda cada ponto para baixo até o múltiplo mais próximo de bucket_size"""
    return bucket_size * np.floor(np.asarray(points, dtype=float) / bucket_size)

def make_histogram(points: np.ndarray, bucket_size: float) -> Tuple[np.ndarray, np.ndarray]:
    """Distribui os pontos nos buckets e conta quantos caem em cada um"""
    buckets, counts = np.unique(bucketize(points, bucket_size), return_counts=True)
    return buckets, counts

assert np.array_equal(bucketize([-3.2, 0.5, 7.9, 10.0], 5), [-5, 0, 5, 10])

`bucketize` recebe o array inteiro: `np.floor` arredonda cada elemento, e a divisão e a multiplicação por `bucket_size` valem para todos de uma vez — é o *broadcasting* com escalar do bloco acima. `np.unique(..., return_counts=True)` devolve os valores distintos, em ordem, e quantas vezes cada um aparece: dois arrays alinhados, um bucket e a contagem dele na mesma posição.

> **🟩 Exemplo**
>
> Repare que `make_histogram` é a mesma ideia do histograma de notas do [Capítulo 3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap03/02-graficos-de-barras.html): contar quantos valores caem em cada piso de bucket. Lá a contagem era um `Counter` sobre `grade // 10 * 10`; aqui é `np.unique` com `return_counts=True` sobre um array já arredondado, e o arredondamento vira uma função nomeada, `bucketize` — o que compensa quando o tamanho do bucket muda de um gráfico para o outro, como acontece logo abaixo.

Para desenhar, basta um gráfico de barras — o mesmo `plt.bar` do Capítulo 3:

In [ ]:
from matplotlib import pyplot as plt

def plot_histogram(points: np.ndarray, bucket_size: float, title: str = ""):
    buckets, counts = make_histogram(points, bucket_size)
    plt.bar(buckets, counts, width=bucket_size)
    plt.xlabel("valor")
    plt.ylabel("frequência")
    plt.title(title)

Considere dois conjuntos de dados fictícios: um uniforme entre -100 e 100, e um normal com média 0 e desvio padrão 57. Para o segundo, precisamos da inversa da normal acumulada — dada uma probabilidade, ela devolve o valor abaixo do qual a normal padrão cai com aquela probabilidade. Chama-se `inverse_normal_cdf` e vem de `scratch_np.probability` — a versão em arrays da função do Grus, que aceita um array de probabilidades e devolve um array de valores.

In [ ]:
from scratch_np.probability import inverse_normal_cdf

def random_normal(n: int, rng: np.random.Generator) -> np.ndarray:
    """n amostras da normal padrão"""
    return inverse_normal_cdf(rng.random(n))

rng = np.random.default_rng(0)

# uniforme entre -100 e 100
uniform = 200 * rng.random(10000) - 100

# normal com média 0, desvio padrão 57
normal = 57 * random_normal(10000, rng)

f"médias: {uniform.mean():.2f} e {normal.mean():.2f}"

`np.random.default_rng(0)` cria um **gerador** de números aleatórios com a semente 0, e `rng.random(10000)` sorteia dez mil uniformes em `[0, 1)` de uma vez. Todo sorteio deste livro sai de um gerador assim, criado no chunk e passado explicitamente para quem sorteia — `random_normal` recebe o `rng` como parâmetro em vez de puxar de um estado global —, para que a semente fique visível onde o sorteio acontece. Sem ela, cada execução desenharia dois histogramas diferentes, e os números que esta seção afirma não seriam os seus.

Ambos têm média perto de 0. O histograma mostra o quanto isso esconde — comece pela uniforme:

In [ ]:
# Figura: Histograma da distribuição uniforme
plot_histogram(uniform, 10, "Histograma — uniforme")
plt.show()

Escrever `bucketize` uma vez vale a pena: é assim que se enxerga o que um histograma decide por você. Escrevê-lo toda vez, não. Do jeito que se faz no trabalho, as duas amostras vão para uma tabela — uma coluna cada — e o resumo sai de uma chamada:

In [ ]:
amostras = pd.DataFrame({"uniforme": uniform, "normal": normal})
amostras.describe().round(2)

Contagem, média, desvio, mínimo, quartis e máximo, para as duas colunas, numa chamada — o `describe` que o [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) apresentou. E o resumo confirma o que a seção quer dizer: as duas amostras têm média perto de 0 e desvio padrão perto de 57 (57,81 e 56,66). Se a exploração parasse aqui, as duas distribuições seriam a mesma coisa.

Não são, e o histograma mostra. `hist()` desenha um por coluna:

In [ ]:
# Figura: As mesmas duas amostras, lado a lado — resumos quase idênticos, formas opostas
amostras.hist(bins=20, figsize=(9, 3.5), layout=(1, 2))
plt.show()

Um é achatado e limitado; o outro tem pico ao centro e caudas compridas que chegam a −215 e 198 — valores que a uniforme não alcança nem por acidente. Nenhuma estatística resumo diz isso; a tabela do `describe` acima, olhada com atenção, no máximo insinua (repare no mínimo e no máximo). Só a forma diz.

### Duas dimensões

Com duas dimensões, você quer entender cada uma individualmente, mas também como elas se relacionam. Considere mais um conjunto fictício:

In [ ]:
rng = np.random.default_rng(4)
xs = random_normal(1000, rng)
ys1 =  xs + random_normal(1000, rng) / 2
ys2 = -xs + random_normal(1000, rng) / 2

Se você rodasse `plot_histogram` em `ys1` e `ys2` separadamente, teria dois histogramas parecidos — de fato, os dois têm distribuição normal, com a mesma média e o mesmo desvio padrão. Mas cada um tem uma relação muito diferente com `xs`, e isso só aparece quando você olha as duas dimensões juntas:

In [ ]:
# Figura: Duas distribuições conjuntas muito diferentes
plt.scatter(xs, ys1, marker='.', color='black', label='ys1')
plt.scatter(xs, ys2, marker='.', color='gray',  label='ys2')
plt.xlabel('xs')
plt.ylabel('ys')
plt.legend(loc=9)
plt.title("Distribuições conjuntas muito diferentes")
plt.show()

`ys1` cresce com `xs`; `ys2` decresce. A diferença fica evidente também na correlação. `df.corr()` devolve a matriz de correlações entre todas as colunas, com 1 na diagonal e o nome de cada série na margem — não é preciso lembrar qual índice era qual:

In [ ]:
pd.DataFrame({"xs": xs, "ys1": ys1, "ys2": ys2}).corr().round(3)

0,893 contra −0,897: a mesma força, sinais opostos. E a terceira entrada, −0,806 entre `ys1` e `ys2`, sai de graça — é o tipo de pergunta que não se faz quando cada correlação custa uma chamada.

> **⚠️ Atenção — Uma afirmação sobre dado aleatório precisa de semente**
>
> Suponha que você quisesse registrar esse resultado como um teste, do jeito que este livro faz o tempo todo:
>
> ```python
> assert 0.89 < np.corrcoef(xs, ys1)[0, 1] < 0.91
> assert -0.91 < np.corrcoef(xs, ys2)[0, 1] < -0.89
> ```
>
> Se `xs`, `ys1` e `ys2` fossem gerados **sem** fixar semente, esse teste seria cara ou coroa. A correlação dessa construção gira em torno de 0,894 — perto o bastante da borda inferior da janela `(0.89, 0.91)` para que o `assert` falhe cerca de uma vez em cada quatro execuções (medido: 24,6% e 25,0% em duas séries independentes de 20 mil repetições do código do Grus, e 24,6% em mais 20 mil com o gerador do `numpy` — a distribuição é a mesma). Não seria um defeito no cálculo da correlação: é uma afirmação sobre uma amostra aleatória, escrita como se fosse determinística.
>
> São duas lições, e valem para qualquer código seu: uma afirmação sobre dado aleatório só é reprodutível se a semente estiver fixada, e uma janela de tolerância só é segura se for larga o bastante para a variação real da amostra, não só para o valor obtido numa execução específica. Por isso `rng = np.random.default_rng(4)` aparece aqui em cima, antes de `xs`: com a semente fixada, a correlação sai sempre a mesma — 0,893 e -0,897, a três milésimos da borda, mas sempre os mesmos três milésimos — em qualquer execução.

### Muitas dimensões

Com muitas dimensões, você quer saber como todas elas se relacionam entre si. Uma abordagem simples é a **matriz de correlação**, cuja entrada na linha $i$, coluna $j$ é a correlação entre a dimensão $i$ e a dimensão $j$ dos dados:

In [ ]:
def correlation_matrix(X: np.ndarray) -> np.ndarray:
    """
    Devolve a matriz d x d cuja entrada (i, j) é a correlação
    entre a coluna i e a coluna j de X
    """
    return np.corrcoef(X, rowvar=False)

A função tem uma linha, e ela existe por causa de um detalhe que morde: `np.corrcoef` assume, por padrão, uma **variável por linha** — é o `rowvar=True` implícito. A convenção deste livro é o contrário, uma variável por coluna, e é o `rowvar=False` que diz isso. Esquecê-lo não dá erro: dá uma matriz $n \times n$ de correlações entre observações, um número que não significa nada, calculado sem aviso. É uma armadilha que só existe do lado do array. Num `DataFrame` não há o que confundir: coluna **é** variável, por definição, e `df.corr()` não tem um parâmetro `rowvar` porque não precisa de um. O preço de perder os nomes, na fronteira, é ter de lembrar da convenção sozinho.

Uma abordagem mais visual — quando o número de dimensões não é grande demais — é a **matriz de dispersão**, mostrando todos os gráficos de dispersão dois a dois. Para gerar um exemplo com correlações interessantes, construímos quatro séries relacionadas entre si de propósito — cada uma um array de 100 valores, que viram as quatro colunas de uma tabela. `np.where(cond, 6, 0)` escolhe, elemento a elemento, 6 onde a máscara é verdadeira e 0 onde não é: o `6 if ... else 0` do Grus, para a coluna inteira. Desenhar a grade é uma chamada — `pd.plotting.scatter_matrix` faz os dezesseis painéis, põe um histograma de cada série na diagonal e escreve os nomes nos eixos.

In [ ]:
# Figura: Matriz de dispersão de quatro séries correlacionadas
rng = np.random.default_rng(0)
num_points = 100

col0 = random_normal(num_points, rng)
col1 = -5 * col0 + random_normal(num_points, rng)
col2 = col0 + col1 + 5 * random_normal(num_points, rng)
col3 = np.where(col2 > -2, 6, 0)

series = pd.DataFrame({"serie0": col0, "serie1": col1,
                       "serie2": col2, "serie3": col3})

pd.plotting.scatter_matrix(series, figsize=(8, 8))
plt.show()

In [ ]:
print(np.round(correlation_matrix(series.to_numpy()), 2))   # a matriz, sem nomes

series.corr().round(2)                                      # a mesma, com nomes

A matriz confirma o que os gráficos sugerem: a série 1 é fortemente **anticorrelacionada** com a série 0 (a própria fórmula de `col1` inverte o sinal de `col0`); a série 2 é positivamente correlacionada com a série 1; e a série 3 só assume os valores 0 e 6 — 6 quando a série 2 é grande, 0 quando é pequena —, o que explica as duas faixas de pontos nos gráficos que a envolvem: verticais quando a série 3 está no eixo $x$, horizontais quando está no eixo $y$. Os mesmos dezesseis números, dos dois lados da fronteira. A diferença é só que num deles é preciso lembrar que a linha 1 era a série 1.

> **🔷 Conceito**
>
> A matriz de dispersão é uma ferramenta de **triagem**, não de conclusão. Ela mostra rápido quais pares de dimensões merecem uma olhada mais de perto — o que é valioso justamente porque examinar cada par manualmente não escala além de um punhado de dimensões. A [seção 7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html) lida com o problema oposto: dimensões demais para sequer desenhar uma matriz dessas.

> **💡 Dica — Na prática: o que `describe`, `corr` e `hist` decidem por você**
>
> Três chamadas desta seção resumiram um conjunto de dados, e cada uma tomou decisões que não pediram sua opinião.
>
> **`.corr()` escolhe um coeficiente.** O padrão é Pearson — o mesmo do `np.corrcoef`, e com a convenção certa, uma variável por coluna. Um parâmetro troca para Spearman ou Kendall, que medem relação *monotônica* em vez de linear e aguentam melhor um outlier. Se as suas séries têm uma relação forte mas curva, Pearson vai chamá-la de fraca, com toda a cara de resposta.
>
> **`.describe()` escolhe as estatísticas.** Ele decide o que mostrar pelo tipo da coluna: para número, o resumo de cinco pontos; para texto, contagem, quantos valores distintos e o mais frequente. É conveniente e é uma armadilha pequena — uma coluna numérica que chegou como texto (o caso do `object` que o [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) mostrou) aparece no `describe` com uma cara plausível e sem média nenhuma.
>
> **`.hist()` e `scatter_matrix` escolhem os baldes.** O número de barras sai de uma heurística — a mesma decisão que você tomou à mão em `bucketize`, com a diferença de que ali ela estava escrita. Um `bucket_size` ruim esconde exatamente o tipo de diferença que a seção *Dados unidimensionais* existe para revelar, e é por isso que `bins=20` aparece na chamada acima: o padrão deixava a uniforme com degraus que sugeriam estrutura onde não há nenhuma.
>
> Fora deste livro, quem faz muito disso costuma usar o [`seaborn`](https://seaborn.pydata.org/), construído em cima do `matplotlib` e do `pandas`: `seaborn.pairplot(df)` é a matriz de dispersão com dispersão e densidade já escolhidas, e `seaborn.heatmap(df.corr())` desenha a matriz de correlação com cor em vez de número. Nenhum dos dois é dependência deste livro; os dois valem o tempo de instalar quando a exploração for o trabalho, e não o preâmbulo dele.

## Usando NamedTuples

> **📌 Nota**
>
> Esta seção corresponde a *Using NamedTuples*, do capítulo 10 de Grus (2019).

O bloco *De listas a arrays* da [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) deixou claro o que um array não faz: guardar um `str`, uma data e um `float` lado a lado — o `dtype` é um só. Um registro heterogêneo pede outra estrutura, e a forma mais comum de representá-lo é um `dict`:

In [ ]:
import datetime

stock_price = {'closing_price': 102.06,
               'date': datetime.date(2014, 8, 29),
               'symbol': 'AAPL'}

Existem boas razões para isso ser menos ideal do que parece.

A primeira é desempenho: um `dict` carrega overhead que uma estrutura mais enxuta evitaria — mas, na maioria dos casos, isso é secundário perto do problema seguinte.

A segunda razão é mais séria: acessar campos por chave de `dict` é propenso a erro. O código abaixo roda **sem** erro nenhum, e faz a coisa errada silenciosamente:

In [ ]:
# opa, erro de digitação
stock_price['cosing_price'] = 103.06
'closing_price' in stock_price, 'cosing_price' in stock_price

`stock_price` agora tem duas chaves parecidas, `closing_price` e `cosing_price`, e nada avisou. Qualquer código que dependa de `stock_price['closing_price']` continua lendo o valor antigo (102.06), enquanto o valor novo (103.06) fica escondido atrás de uma chave com erro de digitação.

Por fim, embora seja possível anotar o tipo de um `dict` uniforme —

In [ ]:
from typing import Dict
prices: Dict[datetime.date, float] = {}

— não existe um jeito útil de anotar um `dict` que representa dado heterogêneo, como o `stock_price` acima, que mistura `str`, `datetime.date` e `float` num único registro. A anotação de tipo perde exatamente o poder que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) apresentou: dizer, antes de ler o corpo do código, o que cada campo é.

### `namedtuple`

Python tem uma classe embutida para isso: `namedtuple`, que é como uma tupla, mas com posições nomeadas.

In [ ]:
from collections import namedtuple

StockPrice = namedtuple('StockPrice', ['symbol', 'date', 'closing_price'])
price = StockPrice('MSFT', datetime.date(2018, 12, 14), 106.03)

assert price.symbol == 'MSFT'
assert price.closing_price == 106.03

Como tuplas comuns, `namedtuple`s são **imutáveis** — não dá para modificar um valor depois de criado. Isso às vezes atrapalha, mas na maior parte das vezes é uma vantagem: um registro que não muda por baixo dos seus pés é mais fácil de raciocinar sobre.

O que `namedtuple` ainda não resolve é a anotação de tipo — `StockPrice` acima não diz, em lugar nenhum, que `closing_price` é `float`. É para isso que existe a variante tipada, `NamedTuple`, do módulo `typing`:

In [ ]:
from typing import NamedTuple

class StockPrice(NamedTuple):
    symbol: str
    date: datetime.date
    closing_price: float

    def is_high_tech(self) -> bool:
        """É uma classe — então também dá para acrescentar métodos"""
        return self.symbol in ['MSFT', 'GOOG', 'FB', 'AMZN', 'AAPL']

price = StockPrice('MSFT', datetime.date(2018, 12, 14), 106.03)

assert price.symbol == 'MSFT'
assert price.closing_price == 106.03
assert price.is_high_tech()

Agora `price.clo` faria seu editor sugerir `closing_price` — porque o editor sabe, pela anotação, que esse campo existe e o que ele é. Um `dict` nunca dá essa pista: `stock_price['clo` não tem como o editor completar, porque chaves de `dict` não são parte da assinatura do tipo.

> **❗ Importante**
>
> `price.cosing_price = 103.06` — o mesmo erro de digitação de antes — agora levanta `AttributeError` na hora, em vez de criar silenciosamente um segundo campo. É a mesma classe de proteção que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) atribuiu às anotações de tipo em geral: elas não impedem todo erro, mas tornam alguns deles impossíveis de passar despercebidos.

> **🟩 Exemplo**
>
> Você vai ver este exato padrão de novo no [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html), sobre Naive Bayes:
>
> ```python
> class Message(NamedTuple):
>     text: str
>     is_spam: bool
> ```
>
> `Message` empacota um texto e o rótulo dele exatamente como `StockPrice` empacota um símbolo, uma data e um preço — um registro pequeno, imutável, com nomes de campo que o editor entende. É o uso mais comum de `NamedTuple` neste livro: dar nome e tipo ao que fica **fora** dos arrays — o texto, o rótulo, a data —, enquanto o que é número vai para dentro deles.

> **💡 Dica — Na prática: o que se faz com isso**
>
> `NamedTuple` resolve o problema do registro heterogêneo tipado, mas continua imutável — o que é uma escolha, não uma limitação a se contornar. Mutabilidade, sem abrir mão da anotação de tipo, é o assunto da [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/03-dataclasses.html): `dataclass` — que fecha apontando para um terceiro caminho, o de quando os registros não vêm um por vez, mas aos milhares.

## Dataclasses

> **📌 Nota**
>
> Esta seção corresponde a *Dataclasses*, do capítulo 10 de Grus (2019).

`dataclass` é, mais ou menos, uma versão mutável de `NamedTuple`. "Mais ou menos" porque a semelhança é de propósito, não de implementação: um `NamedTuple` representa o dado como uma tupla por baixo dos panos, enquanto uma `dataclass` é uma classe Python normal para a qual o decorador gera automaticamente alguns métodos — `__init__`, `__repr__`, `__eq__` — que você teria que escrever à mão.

> **❗ Importante**
>
> `dataclass` existe desde o Python 3.7 — código que usa `@dataclass` não roda num interpretador mais antigo que isso.

A sintaxe é parecida com a de `NamedTuple`. Em vez de herdar de uma classe base, usa-se um decorador:

In [ ]:
import datetime
from dataclasses import dataclass

@dataclass
class StockPrice2:
    symbol: str
    date: datetime.date
    closing_price: float

    def is_high_tech(self) -> bool:
        """É uma classe, então também dá para acrescentar métodos"""
        return self.symbol in ['MSFT', 'GOOG', 'FB', 'AMZN', 'AAPL']

price2 = StockPrice2('MSFT', datetime.date(2018, 12, 14), 106.03)

assert price2.symbol == 'MSFT'
assert price2.closing_price == 106.03
assert price2.is_high_tech()

A diferença que importa: dá para modificar os valores de uma instância de `dataclass`.

In [ ]:
# desdobramento de ações
price2.closing_price /= 2
assert price2.closing_price == 106.03 / 2
price2.closing_price

Tentar a mesma coisa com a versão `NamedTuple` da seção anterior levantaria `AttributeError` — campos de `NamedTuple` não têm `setter`, porque por baixo continuam sendo posições de uma tupla.

Só que a mutabilidade reabre exatamente o problema que motivou trocar `dict` por `NamedTuple`, na seção anterior: como `StockPrice2` é uma classe normal, Python deixa você criar atributos novos nela em qualquer momento, sem checagem nenhuma.

In [ ]:
# é uma classe normal — então dá para acrescentar campos à vontade!
price2.cosing_price = 75   # opa

hasattr(price2, 'cosing_price'), hasattr(price2, 'closing_price')

O erro de digitação de duas seções atrás — `cosing_price` em vez de `closing_price` — volta a passar sem aviso nenhum. `dataclass` resolve o problema de desempenho e o de anotação de tipo do `dict`, mas não o de digitação: essa proteção específica é exclusiva do `NamedTuple`, porque tuplas não aceitam atributo novo de jeito nenhum.

### O terceiro caminho: uma linha de tabela

`dict`, `NamedTuple` e `dataclass` resolvem o mesmo problema: representar **um** registro heterogêneo. A pergunta que nenhum dos três responde bem é a seguinte: e quando são vinte e três mil registros, e a pergunta é sobre o conjunto deles?

Uma lista de `NamedTuple` responde, mas cada pergunta vira um laço. É para esse caso que existe o **`DataFrame`** do [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) — que é, visto deste ângulo, uma coleção de registros virada de lado: em vez de $n$ objetos com $d$ campos cada, $d$ colunas com $n$ valores cada, uma por tipo. A conversão é direta, e o `pandas` reconhece os nomes dos campos:

In [ ]:
import pandas as pd
from typing import NamedTuple

class StockPrice(NamedTuple):
    symbol: str
    date: datetime.date
    closing_price: float

precos = [StockPrice('MSFT', datetime.date(2018, 12, 14), 106.03),
          StockPrice('AAPL', datetime.date(2018, 12, 14), 165.48),
          StockPrice('MSFT', datetime.date(2018, 12, 13), 109.42)]

df = pd.DataFrame(precos)

print(df.dtypes)
df

Os nomes dos campos viraram nomes de coluna, e cada coluna ganhou um tipo. Repare que `date` ficou `object`, não `datetime64`: o `DataFrame` está guardando os objetos `datetime.date` que você criou, sem convertê-los. A inferência de tipo do [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) acontece na **leitura** de um arquivo, quando tudo é texto e alguém precisa adivinhar; aqui não havia nada a adivinhar.

E o caminho de volta existe. Percorrer as linhas de um `DataFrame` devolve — sem nenhuma ironia — namedtuples:

In [ ]:
for linha in df.itertuples():
    print(linha.symbol, linha.closing_price)

`linha.symbol` e `linha.closing_price`, com o editor completando os nomes e o erro de digitação levantando `AttributeError` — as três vantagens que a [seção 7.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/02-namedtuples.html) atribuiu ao `NamedTuple`, intactas. O `DataFrame` não substitui o registro tipado; ele o guarda de outro jeito e o devolve quando você pede uma linha.

A escolha entre os dois, então, não é de gosto: é sobre **de que lado da pergunta você está**. Uma linha por vez — o resultado de um `parse`, o registro que chegou de uma API, a mensagem que vai para o classificador do [Capítulo 10](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap10/index.html) — pede `NamedTuple`. Todas as linhas de uma vez — quantas por símbolo, qual a média por mês, quais estão fora da faixa — pede `DataFrame`. E é por isso que a [seção 7.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/05-manipulando-dados.html), que faz exatamente essas perguntas, é escrita em `pandas` — com um laço à mão, uma vez, para o leitor ver o que o `groupby` faz por baixo.

> **💡 Dica — Na prática: o que se faz com isso**
>
> `dataclass` aparece bastante em código Python fora deste livro, sobretudo em lugares onde mutabilidade é mesmo necessária — um objeto que representa configuração sendo montada aos poucos, por exemplo, ou o estado interno de alguma coisa que muda ao longo do tempo. Onde o registro é fixo desde a criação — a leitura de uma linha de um CSV, o resultado de uma consulta —, `NamedTuple` continua sendo a escolha mais restritiva, e "mais restritivo" aqui é elogio: menos formas de errar.
>
> Bibliotecas de validação de dado, como `pydantic`, cobrem o caso em que nem `NamedTuple` nem `dataclass` bastam: uma classe com campos tipados que, ao ser construída, *verifica* os tipos de verdade — ao contrário da anotação pura de Python, que é só documentação (o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) mostrou isso: `add(10, "cinco")` com `a: int, b: int` não impede nada). `pydantic` é o que sustenta boa parte das APIs web modernas em Python — frameworks como FastAPI o usam para transformar um JSON de entrada em um objeto tipado, rejeitando a requisição se um campo vier com o tipo errado.

## Limpeza e Transformação

> **📌 Nota**
>
> Esta seção corresponde a *Cleaning and Munging*, do capítulo 10 de Grus (2019).

Dado do mundo real é **sujo**. Você geralmente vai ter que trabalhar nele antes de conseguir usá-lo: converter strings para `float` ou `int`, checar valores ausentes, tratar outliers e dado corrompido. O [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) já mostrou exemplos disso, ao ler linhas de arquivo que vêm sempre como texto puro.

Uma opção é fazer a conversão bem antes de usar o dado:

```python
closing_price = float(row[2])
```

Mas é provavelmente menos sujeito a erro fazer o *parsing* dentro de uma função que dá para testar isoladamente:

In [ ]:
import datetime
from typing import List, NamedTuple
from dateutil.parser import parse

class StockPrice(NamedTuple):
    symbol: str
    date: datetime.date
    closing_price: float

def parse_row(row: List[str]) -> StockPrice:
    symbol, date, closing_price = row
    return StockPrice(symbol=symbol,
                      date=parse(date).date(),
                      closing_price=float(closing_price))

# testa a função
stock = parse_row(["MSFT", "2018-12-14", "106.03"])

assert stock.symbol == "MSFT"
assert stock.date == datetime.date(2018, 12, 14)
assert stock.closing_price == 106.03

### E se o dado for ruim?

`parse_row` funciona, mas quebra ao primeiro valor inesperado — uma data mal formatada, um preço que não é número. Talvez você prefira receber `None` no lugar de um `ValueError` estourado no meio do programa:

In [ ]:
from typing import Optional
import re

def try_parse_row(row: List[str]) -> Optional[StockPrice]:
    symbol, date_, closing_price_ = row

    # o símbolo da ação deveria ser só letras maiúsculas
    if not re.match(r"^[A-Z]+$", symbol):
        return None

    try:
        date = parse(date_).date()
    except ValueError:
        return None

    try:
        closing_price = float(closing_price_)
    except ValueError:
        return None

    return StockPrice(symbol, date, closing_price)

# deveria devolver None para erros
assert try_parse_row(["MSFT0", "2018-12-14", "106.03"]) is None  # símbolo ruim
assert try_parse_row(["MSFT", "2018-12--14", "106.03"]) is None  # data ruim
assert try_parse_row(["MSFT", "2018-12-14", "x"]) is None        # preço ruim

# mas deveria devolver o mesmo de antes se o dado é bom
assert try_parse_row(["MSFT", "2018-12-14", "106.03"]) == stock

Agora podemos ler um arquivo real com dado sujo — `dados/comma_delimited_stock_prices.csv`, que tem exatamente esse problema:

In [ ]:
with open("dados/comma_delimited_stock_prices.csv") as f:
    print(f.read())

In [ ]:
import csv

data: List[StockPrice] = []

with open("dados/comma_delimited_stock_prices.csv") as f:
    reader = csv.reader(f)
    for row in reader:
        maybe_stock = try_parse_row(row)
        if maybe_stock is None:
            print(f"pulando linha inválida: {row}")
        else:
            data.append(maybe_stock)

len(data), data

Só uma linha é rejeitada: `MSFT,6/19/2014,n/a`, porque `"n/a"` não converte para `float`. As outras cinco entram — inclusive a da FB com data `6/20/3014`.

> **⚠️ Atenção — O erro que passa pelo filtro**
>
> Repare: `try_parse_row` não rejeitou a linha da FB, e o ano da data dela é **3014**. `"6/20/3014"` é uma string sintaticamente válida — `dateutil.parser.parse` converte sem reclamar, porque é um ano válido do ponto de vista do calendário, só que a mil anos de distância do resto do conjunto. O regex do símbolo não pega isso, e o `try/except ValueError` também não, porque não houve exceção nenhuma.
>
> É o mesmo tipo de outlier que a [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) existe para achar — por inspeção visual ou checagem *ad hoc* —, não algo que uma função de *parsing* bem escrita consegue capturar sozinha. Dado do mundo real tem casas decimais faltando, zeros a mais, erros de digitação e outros problemas do tipo — capturá-los é trabalho seu (talvez não seja oficialmente seu trabalho, mas quem mais vai fazer?).
>
> Guarde essa linha da FB: o ano 3014 volta mais adiante nesta seção, e do outro lado — é ele que faz o `pandas` desistir da coluna inteira sem dizer nada.

> **📌 Nota — O mesmo arquivo pelo `np.genfromtxt`**
>
> A [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) mostrou o leitor tolerante do `numpy`. Sobre este arquivo, ele lê a coluna de preços assim:

In [ ]:
import numpy as np

np.genfromtxt("dados/comma_delimited_stock_prices.csv", delimiter=",", usecols=2)

> O `n/a` virou `nan`, sem aviso — a linha que `try_parse_row` rejeitou em voz alta sobreviveu em silêncio. E o ano 3014 nem foi lido: `usecols=2` descartou a coluna da data antes de qualquer conferência. É o contraste inteiro desta seção: o array carrega os números; a decisão sobre o que é dado ruim continua sendo uma função sua.

E então é preciso decidir o que fazer com as linhas inválidas. Em geral, as três opções são: descartá-las, voltar à fonte e tentar corrigir o dado ausente ou ruim, ou não fazer nada e torcer. Se é uma linha ruim entre milhões, provavelmente está tudo bem ignorá-la. Mas se metade das suas linhas têm dado ruim, isso é algo que precisa ser corrigido.

### O mesmo arquivo com o `pandas`

Nada disso é trabalho que se faça à mão duas vezes. O [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) apresentou o `read_csv`, e ele lê este arquivo sem uma linha de laço — o arquivo não tem cabeçalho, então os nomes das colunas vão no argumento `names`:

In [ ]:
import pandas as pd

precos = pd.read_csv("dados/comma_delimited_stock_prices.csv",
                     names=["symbol", "date", "closing_price"])

print(precos.dtypes)
print(precos.isna().sum())
precos

Duas diferenças em relação ao laço, e as duas importam.

A primeira: **as seis linhas estão lá**. `try_parse_row` descartou a linha `MSFT,6/19/2014,n/a` e imprimiu um aviso; o `read_csv` a manteve, com `NaN` no lugar do preço. O `"n/a"` está na lista de marcadores de ausência que o `pandas` já conhece — `NA`, `N/A`, `n/a`, `null`, `NaN`, campo vazio —, então nem foi preciso pedir. É a coerção silenciosa: nenhuma linha "pulando linha inválida", nenhuma exceção, e uma coluna `float64` com um buraco dentro.

A segunda: **`isna().sum()` é o que substitui aquele aviso impresso**. Ele conta os ausentes por coluna, e é o comando a rodar logo depois de todo `read_csv` — a informação que o laço dava de graça, e que aqui você tem de ir buscar. Uma ausência em `closing_price`, nenhuma em `date`… o que nos leva ao problema seguinte.

A coluna `date` continua sendo texto — `object` no `dtypes` acima. O jeito de pedir a conversão é o argumento `parse_dates`, o mesmo do Capítulo 6:

In [ ]:
precos = pd.read_csv("dados/comma_delimited_stock_prices.csv",
                     names=["symbol", "date", "closing_price"],
                     parse_dates=["date"])

precos.dtypes

`date` continua `object`. **Você pediu a conversão e ela não aconteceu** — sem erro, sem aviso, sem uma linha de saída dizendo que algo deu errado. O único sinal é o `dtypes`, e só para quem foi olhar.

O culpado é a linha da FB. Uma data do `pandas` é um `datetime64[ns]`, um número de nanossegundos desde 1970, e um inteiro de 64 bits de nanossegundos cobre de 1677 a 2262 — `pd.Timestamp.max` é `2262-04-11`. O ano **3014** não cabe. Diante de uma coluna que ele não consegue converter inteira, o `read_csv` desiste dela e devolve o texto original, calado. Uma linha ruim em seis contaminou as seis.

A mesma conversão, pedida diretamente, diz o que aconteceu:

In [ ]:
try:
    pd.to_datetime(precos["date"])
except Exception as erro:
    print(type(erro).__name__)
    print(str(erro).split("You might")[0].strip())

Posição 2, a linha da FB, com o valor no texto do erro. É a informação que o `read_csv` tinha e engoliu.

E aqui a seção se inverte: **o `pandas` pegou o outlier que a nossa função deixou passar.** `try_parse_row` aceitou `6/20/3014` porque o `dateutil` converte sem reclamar — é um ano válido do calendário — e o regex do símbolo não tinha nada a dizer sobre datas. O `pandas` reclamou por um motivo que não tem nada a ver com cuidado: o tipo dele não cabe o ano 3014. Não é que uma ferramenta seja mais segura que a outra; é que **cada uma tem a sua noção de dado ruim**, e nenhuma das duas é a sua. Saber qual é a noção de cada uma é o trabalho.

Com o problema à vista, dá para decidir o que fazer com ele — que é exatamente a decisão que esta seção listou acima, agora com o dado na mão. `errors="coerce"` converte o que dá e põe `NaT` (*not a time*) no que não dá:

In [ ]:
precos["date"] = pd.to_datetime(precos["date"], errors="coerce")

print(precos.isna().sum())
print(f"{len(precos)} linhas lidas, {len(precos.dropna())} sobrevivem")
precos.dropna()

Quatro linhas, não cinco. O laço com `try_parse_row` descartou **uma** — a do `n/a` — e entregou cinco registros, com o ano 3014 entre eles. O caminho do `pandas` descartou **duas**, porque conta o `NaT` da data impossível como ausência igual à do preço.

Os índices que sobraram — 0, 1, 3, 5 — dizem de que linha do arquivo veio cada uma. É o índice do `DataFrame`, que não é a posição: `dropna()` tirou linhas e não renumerou nada, o que é uma vantagem quando se precisa voltar à fonte e ver o que havia ali.

E o `dropna()` é uma das três opções que esta seção listou acima, não a resposta certa. Ele é a mais fácil de escrever e a mais fácil de escrever **sem pensar** — é o `.dropna()` no fim de uma linha comprida, que some no meio do código e leva junto vinte por cento do conjunto sem que ninguém veja. É por isso que o `print` acima existe: **conte antes e depois, sempre**. Uma linha ruim em um milhão se ignora; metade das linhas ruins é um problema de fonte de dados, e nenhum `dropna()` conserta isso.

> **💡 Dica — Na prática: quatro leitores, quatro noções de "dado ruim"**
>
> Esta seção passou o mesmo arquivo de seis linhas por quatro caminhos. Vale ver o balanço, porque nenhum deles é a resposta certa e todos aparecem em código de verdade:
>
> | leitor | `MSFT,…,n/a` | `FB,6/20/3014,…` | você fica sabendo? |
> |---|---|---|---|
> | `try_parse_row` | descarta a linha | **aceita** | sim, imprime a linha pulada |
> | `pd.read_csv` | vira `NaN` | desiste da coluna de datas | só no `dtypes` e no `isna()` |
> | `pd.to_datetime` | — | levanta `OutOfBoundsDatetime` | sim, com a posição |
> | `np.genfromtxt` | vira `nan` | não foi lido (`usecols`) | não |
>
> Duas coisas que a tabela deixa implícitas e que vale dizer em voz alta:
>
> **`errors="coerce"` é uma decisão, não um conserto.** O nome é bom: ele *coage*. Onde `try_parse_row` devolve `None` e a linha é descartada com um aviso impresso, `coerce` não descarta nada — converte o valor problemático em `NaN` (ou `NaT`) e a linha *sobrevive* na tabela. Ela não aparece como "pulando linha inválida" em lugar nenhum; ela aparece depois, quando alguém calcula uma média sem checar `.isna()` antes, e o resultado sai errado sem erro nenhum.
>
> **O silêncio do `parse_dates` é o pior dos quatro comportamentos**, e não porque o `pandas` seja descuidado: converter uma coluna inteira é tudo ou nada, e desistir é a única saída que não inventa dado. O que se pode fazer contra isso é o hábito de sempre: `dtypes` e `isna().sum()` depois de toda leitura, e desconfiar de uma coluna `object` que devia ser data ou número. É o mesmo aviso do [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html), e ele é repetido aqui porque esta seção é a única do livro em que ele custa caro na frente do leitor.
>
> Fora deste livro há uma família de ferramentas que existe só para isso — `pandera`, `great-expectations` e parentes: você declara o que a coluna deveria ser (faixa, tipo, unicidade, "nenhuma data depois de hoje") e a validação falha alto, na entrada, em vez de silenciosamente, no meio. É a ideia do `assert` deste livro, aplicada a uma tabela inteira. Um `assert pd.to_datetime(precos["date"], errors="coerce").notna().all()` teria pego a linha da FB — na entrada, e em voz alta.

## Manipulando Dados

> **📌 Nota**
>
> Esta seção corresponde a *Manipulating Data*, do capítulo 10 de Grus (2019).

Uma das habilidades mais importantes de quem trabalha com dado é **manipulá-lo**. É mais uma abordagem geral do que uma técnica específica — então esta seção percorre alguns exemplos, para dar a ideia.

A [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-limpeza-e-transformacao.html) tratou de seis linhas. Esta trata de vinte e três mil, e a mudança de escala muda a ferramenta: cada pergunta aqui é sobre o **conjunto** — o maior preço de cada ação, a maior variação de um dia para o outro, a média por mês —, e nenhuma delas se responde bem percorrendo registros um a um. É o caso do `DataFrame`, e é o que a [seção 7.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/03-dataclasses.html) antecipou.

O arquivo é o `dados/stocks.csv` inteiro: 23.105 preços diários de quatro ações (AAPL, FB, GOOG, MSFT), o mesmo que o [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) leu. Ao longo do caminho, vamos notar padrões no que estamos fazendo e abstrair ferramentas que tornem a manipulação mais fácil.

In [ ]:
import numpy as np
import pandas as pd

acoes = pd.read_csv("dados/stocks.csv",
                    usecols=["Symbol", "Date", "Close"],
                    parse_dates=["Date"])

print(acoes.shape)
print(acoes.dtypes)
acoes.head(3)

Uma chamada, três argumentos, e a tabela está pronta. `usecols` fica só com as três colunas que esta seção usa — o arquivo tem oito —, e `parse_dates` faz o que a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-limpeza-e-transformacao.html) mostrou dar errado quando o dado é sujo: aqui dá certo, e o `dtypes` prova, com `Date` em `datetime64[ns]`. Vale conferir sempre; é barato.

Repare no que **não** foi preciso: nenhum laço, nenhuma conversão de campo, nenhuma função de *parsing*, nenhuma decisão sobre linha ruim. As 23.105 linhas deste arquivo são limpas, e o `read_csv` as leu como estão — três colunas, três tipos, uma tabela. É a diferença entre o arquivo de brinquedo da seção anterior, feito para ter defeito, e um arquivo de verdade que por acaso não tem nenhum.

Suponha que queremos saber o maior preço de fechamento já registrado para a AAPL. Podemos dividir isso em passos concretos:

1. Restringir às linhas da AAPL.
2. Pegar o `closing_price` de cada linha.
3. Tirar o `max` desses preços.

Uma máscara faz as três coisas de uma vez, e é a mesma máscara booleana do bloco *De listas a arrays*: `acoes["Symbol"] == "AAPL"` é o filtro — uma coluna de 23.105 booleanos —, `.loc[..., "Close"]` aplica o filtro e escolhe a coluna, e `.max()` é a resposta:

In [ ]:
max_aapl_price = acoes.loc[acoes["Symbol"] == "AAPL", "Close"].max()
max_aapl_price

`.loc` recebe duas coisas separadas por vírgula: quais linhas e quais colunas. As linhas vêm da máscara, a coluna vem pelo nome — e é por isso que existem `.loc` e `.iloc` separados no `pandas`: um escolhe por **rótulo**, o outro por **posição**. Num array só existe a posição.

Mais geralmente, podemos querer o maior preço de fechamento de cada ação. A receita é a de sempre: listar os símbolos distintos, e para cada um repetir a máscara e o `.max()` — um laço sobre quatro ações, não sobre 23.105 linhas.

In [ ]:
for simbolo in sorted(acoes["Symbol"].unique()):
    maximo = acoes.loc[acoes["Symbol"] == simbolo, "Close"].max()
    print(f"{simbolo}: {maximo:.2f}")

Funciona, e é o padrão que a seção prometeu notar: **um laço sobre os valores distintos de uma coluna, repetindo a mesma conta em cada pedaço**. Esse padrão tem nome e tem uma linha:

In [ ]:
acoes.groupby("Symbol")["Close"].max()

`groupby("Symbol")` parte a tabela nos quatro pedaços que o laço percorria; `["Close"]` escolhe a coluna; `.max()` é aplicado a cada pedaço. É literalmente o laço de cima — máscara, coluna, redução —, e é assim que vale lê-lo sempre que ele aparecer: **`groupby` é o laço sobre os valores distintos, escrito de uma vez**. Daqui em diante usamos a linha; o laço já cumpriu o papel de mostrar o que ela faz.

Repare que os símbolos saíram em ordem alfabética. O `groupby` ordena pela chave por padrão — foi por isso que o laço acima percorreu `sorted(...)`, para as duas saídas serem comparáveis linha a linha. E o resultado é uma **`Series`** com o símbolo no índice, não uma tabela com uma coluna de símbolos: o que era chave do agrupamento vira rótulo.

Agora podemos fazer perguntas mais complicadas, como quais foram as maiores e as menores variações percentuais em um único dia. A variação é `preço_hoje / preço_ontem - 1`, o que exige associar o preço de hoje ao de ontem — dentro de cada ação, e na ordem certa. Então o primeiro passo é garantir a ordem:

In [ ]:
acoes = acoes.sort_values(["Symbol", "Date"])
acoes.head(3)

O `head(3)` não mudou — a AAPL já era o primeiro bloco do arquivo —, mas a tabela mudou: os blocos vinham em `AAPL, MSFT, FB, GOOG` e agora saem em ordem alfabética. E mesmo que nada tivesse mudado, não seria motivo para tirar a linha, porque **nada no que vem a seguir verifica isso**. A conta de variação diária assume que a linha de baixo é o dia seguinte da mesma ação, e ela assume em silêncio: com as linhas embaralhadas, ela não levanta erro nenhum — devolve, sem uma palavra, "variações diárias" entre dias que não são consecutivos, comparando terça com sexta e a AAPL com ela mesma sete anos depois.

Ordenar antes de agrupar é o tipo de linha que parece supérflua até o dia em que o arquivo chega em outra ordem, e aí ela é a diferença entre um número certo e um número plausível. `sort_values(["Symbol", "Date"])` ordena por símbolo e, dentro de cada símbolo, por data — a mesma coisa que faríamos com dois arrays paralelos e uma ordenação por índices, sem o risco de reindexar um e esquecer o outro.

Com cada ação em ordem, podemos calcular uma sequência de variações dia a dia:

In [ ]:
def day_over_day_changes(closes: np.ndarray) -> np.ndarray:
    """Variações percentuais dia a dia. Assume preços de uma única ação, em ordem."""
    return closes[1:] / closes[:-1] - 1

aapl = acoes.loc[acoes["Symbol"] == "AAPL", "Close"].to_numpy()   # a fronteira
variacoes_aapl = day_over_day_changes(aapl)

len(aapl), len(variacoes_aapl), f"{variacoes_aapl.max():+.2%}"

`closes[1:]` é "hoje" — do segundo dia em diante — e `closes[:-1]` é "ontem" — até o penúltimo. As duas fatias têm o mesmo tamanho e estão desalinhadas por uma posição, que é exatamente o truque de percorrer duas sequências em paralelo do `zip` do [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html); a divisão elemento a elemento faz o resto. Repare no `.to_numpy()`: a função recebe um array porque a conta é sobre números, e é a [fronteira](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) de novo, aqui só para uma coluna. Repare também que saíram 9.583 variações de 9.584 preços — o primeiro dia não tem ontem.

Fazer isso para as quatro ações significaria repetir a fatia dentro de um laço e emendar os pedaços. É outro padrão com nome e com uma linha:

In [ ]:
acoes["variacao"] = acoes.groupby("Symbol")["Close"].pct_change()

print(f"{acoes['variacao'].notna().sum()} variações, "
      f"{acoes['variacao'].isna().sum()} NaN (o primeiro dia de cada ação)")
acoes.head(3)

`pct_change()` é a fatia deslocada, e `groupby("Symbol")` a aplica dentro de cada ação — sem misturar o último dia da AAPL com o primeiro do FB. São 23.101 variações, quatro a menos que linhas.

A diferença em relação à versão à mão é onde essas quatro linhas foram parar. A função devolvia um array **menor**, sem o primeiro dia; a coluna nova tem o mesmo tamanho da tabela, com `NaN` nas quatro linhas que não têm ontem. É uma escolha do `pandas` e é a certa aqui: a variação virou uma coluna ao lado do preço, na mesma linha, e para isso a linha precisa existir.

Com a coluna pronta, achar os extremos é uma chamada — `idxmax` devolve o **rótulo** da linha do maior valor, e `.loc` busca a linha por esse rótulo:

In [ ]:
i_max, i_min = acoes["variacao"].idxmax(), acoes["variacao"].idxmin()

acoes.loc[[i_max, i_min]]

`idxmax` faz para uma coluna o que uma redução do bloco *De listas a arrays* faz para um array — só que devolve um índice, não um valor. E há uma diferença que morde: o `np.argmax`, que você vai encontrar no [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-o-modelo.html), devolve a **posição** no array; `idxmax` devolve o **rótulo** no índice. Aqui os dois coincidem — 4208 é ao mesmo tempo o rótulo e a posição, porque nenhuma linha foi descartada e a AAPL continua sendo o primeiro bloco da tabela —, e é essa coincidência que torna o erro perigoso: basta um `dropna()` como o da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-limpeza-e-transformacao.html), que deixou os rótulos 0, 1, 3 e 5, para os dois números se separarem. Usar um no lugar do outro não dá erro: dá a linha errada. É a mesma distinção que separa `.loc` de `.iloc`, e é a primeira coisa a lembrar quando um resultado do `pandas` vier de uma linha que não faz sentido.

O maior salto de um dia para o outro foi a AAPL, em 6 de agosto de 1997 — mais de 33% em um único pregão. A maior queda foi também a AAPL, em 29 de setembro de 2000 — quase 52% em um dia. As duas datas batem com eventos reais e documentados: o investimento de US$ 150 milhões da Microsoft na Apple, anunciado naquele agosto de 1997, e o alerta de resultados abaixo do esperado que a Apple divulgou em setembro de 2000.

Podemos agora descobrir qual é o melhor mês para investir em ações de tecnologia: basta a variação diária média por mês. Uma coluna de datas guarda os componentes dela atrás do acessório `.dt` — `.dt.month`, `.dt.year`, `.dt.day_name()` —, e o mês extraído serve de chave de agrupamento como qualquer outra coluna serviria:

In [ ]:
variacao_media = acoes.groupby(acoes["Date"].dt.month)["variacao"].mean()
variacao_media.round(5)

In [ ]:
melhor_mes = int(variacao_media.idxmax())
assert melhor_mes == 10   # outubro é o melhor mês

melhor_mes, f"{variacao_media.max():.2%}"

Outubro é o melhor mês para investir, com uma variação diária média de cerca de 0,29% — número que vale hesitar antes de levar a sério: são apenas quatro ações e quase quatro décadas de dados misturados numa única média por mês, sem nenhum teste de significância por trás. É o tipo de padrão fácil de achar e fácil demais de acreditar.

E há um detalhe silencioso nessa média que vale registrar: as quatro linhas com `NaN` estão dentro dos grupos, e `mean()` simplesmente as ignora. O `pandas` tem uma política de ausência embutida em quase toda agregação; o `numpy` não tem — `acoes["variacao"].to_numpy().mean()` devolve `nan`, e essa diferença atravessa a fronteira com você.

Faremos esse tipo de manipulação pelo livro inteiro, em geral sem chamar muita atenção para ela.

> **💡 Dica — Na prática: o que o `groupby` decide sozinho**
>
> Três linhas desta seção substituíram todos os laços que a versão à mão precisaria, e cada uma delas tomou decisões que não estavam escritas.
>
> **`groupby` ordena pela chave.** Por isso os símbolos saíram em ordem alfabética, e não na ordem em que aparecem no arquivo. Com quatro grupos isso é cosmético; com cem mil chaves é tempo, e `groupby(..., sort=False)` existe para isso.
>
> **Toda agregação ignora `NaN`.** `mean()`, `sum()`, `max()` e companhia pulam os ausentes e calculam com o que sobrou — o que às vezes é o que você quer e às vezes é uma média de dez valores apresentada como se fosse de mil. `count()` conta os presentes; `size()` conta as linhas; a diferença entre os dois é quantos faltam. Do lado do `numpy` não há política nenhuma: um `nan` contamina a redução inteira, o que é mais barulhento e, para uma conta de modelo, geralmente melhor.
>
> **`pct_change()` não ordena nada.** Foi por isso que o `sort_values` veio antes, no corpo desta seção. Vale insistir aqui porque o erro é fácil e o resultado é plausível: sem ordenar, ela devolve variações entre linhas vizinhas quaisquer, sem erro nenhum.
>
> **Agregar por mês de calendário é outra coisa.** A média por mês desta seção é por *mês do ano* — todos os outubros juntos, de 1980 a 2018. Se a pergunta fosse "qual foi o fechamento médio da AAPL em cada mês do calendário", a ferramenta seria o `resample`, que exige uma coluna de datas no índice:
>
> ```python
> aapl = acoes[acoes["Symbol"] == "AAPL"].set_index("Date")
> aapl["Close"].resample("ME").mean()
> ```
>
> Repare no `"ME"`, de *month end*: o `"M"` que a maior parte do material antigo da internet usa foi depreciado no `pandas` 2.2 e sai com aviso. `resample` aceita a mesma família de códigos em `"D"` (dia), `"W"` (semana), `"QE"` (fim de trimestre) e `"YE"` (fim de ano), e é a ferramenta certa para série temporal — que é o assunto de um livro inteiro, e não deste capítulo.

## Reescalonamento

> **📌 Nota**
>
> Esta seção corresponde a *Rescaling*, do capítulo 10 de Grus (2019).

Muitas técnicas são sensíveis à **escala** dos dados. Imagine um conjunto com altura e peso de centenas de cientistas de dados, e que você está tentando identificar agrupamentos de porte físico.

Intuitivamente, gostaríamos que agrupamentos representassem pontos próximos entre si, o que significa que precisamos de alguma noção de distância entre pontos. A distância euclidiana do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html) é, em arrays, `np.linalg.norm(a - b)` — o callout de fechamento daquela seção já a mostrava assim —, então uma abordagem natural é tratar pares (altura, peso) como pontos num espaço bidimensional. Considere as pessoas listadas abaixo:

| Pessoa | Altura (polegadas) | Altura (centímetros) | Peso (libras) |
|---|---|---|---|
| A | 63 | 160 | 150 |
| B | 67 | 170,2 | 160 |
| C | 70 | 177,8 | 171 |

Se medirmos altura em polegadas, o vizinho mais próximo de B é A:

In [ ]:
import numpy as np

A = np.array([63, 150.])    # altura em polegadas, peso em libras
B = np.array([67, 160.])
C = np.array([70, 171.])

np.array([np.linalg.norm(A - B), np.linalg.norm(A - C), np.linalg.norm(B - C)]).round(2)

Só que, se medirmos altura em centímetros, o vizinho mais próximo de B passa a ser C:

In [ ]:
A = np.array([160, 150.])   # altura em centímetros, peso em libras
B = np.array([170.2, 160.])
C = np.array([177.8, 171.])

np.array([np.linalg.norm(A - B), np.linalg.norm(A - C), np.linalg.norm(B - C)]).round(2)

> **⚠️ Atenção**
>
> Nada mudou nos dados — A, B e C continuam as mesmas três pessoas. Só a **unidade** de uma das duas dimensões mudou, e isso bastou para inverter qual ponto é "mais próximo" de qual. Qualquer técnica baseada em distância — vizinhos mais próximos, incluído — está exposta a esse problema sempre que as dimensões não são comparáveis entre si.

Obviamente é um problema se trocar a unidade pode mudar resultados desse jeito. Por essa razão, quando as dimensões não são comparáveis entre si, às vezes vamos **reescalonar** nossos dados, de modo que cada dimensão tenha média 0 e desvio padrão 1. Isso efetivamente elimina as unidades, convertendo cada dimensão em "desvios padrão a partir da média".

Para começar, precisamos calcular a média e o desvio padrão de cada **coluna** — a redução por `axis=0` do bloco *De listas a arrays*:

In [ ]:
from typing import Tuple

def scale(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """devolve a média e o desvio padrão de cada coluna"""
    X = np.asarray(X, dtype=float)
    means = X.mean(axis=0)
    stdevs = X.std(axis=0, ddof=1)
    return means, stdevs

vectors = np.array([[-3, -1, 1], [-1, 0, 1], [1, 1, 1]], dtype=float)
means, stdevs = scale(vectors)

assert np.array_equal(means, [-1, 0, 1])
assert np.array_equal(stdevs, [2, 1, 0])
means, stdevs

Repare no `ddof=1`. O `numpy` divide a variância por $n$ por padrão; o desvio padrão amostral, que divide por $n - 1$, é o `ddof=1` (*delta degrees of freedom*) — a mesma fórmula que o Grus usa e que você provavelmente aprendeu. Sem o argumento, `stdevs` sairia `[1.63, 0.82, 0]` e o `assert` cairia: é uma diferença de escala que não avisa. Repare também na terceira coluna: todo vetor tem 1 ali, então o desvio padrão é 0 — não há variação nenhuma para reescalonar.

Com `means` e `stdevs`, podemos criar um novo conjunto de dados:

In [ ]:
def rescale(X: np.ndarray) -> np.ndarray:
    """
    Reescalona os dados de entrada para que cada coluna tenha
    média 0 e desvio padrão 1. (Deixa uma coluna como está se
    o desvio padrão dela é 0.)
    """
    X = np.asarray(X, dtype=float)
    means, stdevs = scale(X)

    rescaled = X.copy()                 # não mexe no original
    varia = stdevs > 0                  # máscara sobre as colunas
    rescaled[:, varia] = (X[:, varia] - means[varia]) / stdevs[varia]
    return rescaled

rescale(vectors)

O laço duplo do Grus — para cada vetor, para cada posição, `if stdevs[i] > 0` — virou uma máscara sobre as **colunas**: `varia` marca as que têm desvio positivo, `X[:, varia]` seleciona só essas, e a subtração e a divisão são o *broadcasting* de um vetor `(d,)` contra uma matriz `(n, d)`. As colunas fora da máscara ficam como estavam na cópia. `X.copy()` importa: sem ele, `rescaled` seria o próprio `X`, e a função alteraria o argumento de quem a chamou.

E, claro, vale testar que `rescale` faz o que achamos que faz:

In [ ]:
means, stdevs = scale(rescale(vectors))
assert np.array_equal(means, [0, 0, 1])
assert np.array_equal(stdevs, [1, 1, 0])

A terceira coluna continua com desvio padrão 0 — `rescale` deliberadamente deixa em paz qualquer dimensão sem variação, em vez de dividir por zero.

### E com um `DataFrame`?

Vale perguntar por que essas duas funções estão em `numpy`, se a mesa de trabalho deste capítulo é o `pandas` — que faz o mesmo cálculo numa expressão:

In [ ]:
import pandas as pd

df = pd.DataFrame(vectors, columns=["a", "b", "c"])

print(df.std())              # ddof=1 por padrão — ao contrário do numpy
(df - df.mean()) / df.std()

Duas coisas nessa saída, e as duas são o argumento.

A primeira é boa: **`DataFrame.std()` já usa `ddof=1`**, o desvio amostral, ao contrário do `np.std`, que usa `ddof=0` e obrigou o argumento explícito em `scale`. Os desvios batem com os do `scale` — 2, 1 e 0 — sem que fosse preciso pedir. Vale saber porque é uma das poucas incompatibilidades de padrão entre as duas bibliotecas, e ela muda o número em silêncio nos dois sentidos.

A segunda é o motivo de preferirmos o array: **a coluna constante virou `NaN`**. Desvio padrão zero, divisão por zero, `NaN` na coluna inteira — sem aviso, e três colunas viram duas quando o modelo for calcular alguma coisa com elas. `rescale` trata esse caso explicitamente, com a máscara `stdevs > 0`, porque a decisão de deixar uma dimensão sem variação em paz é uma decisão de **modelagem**, e o parágrafo abaixo mostra que ela nem sempre é a certa. Uma expressão de uma linha não tem onde guardar essa decisão.

É a divisão de trabalho deste livro em miniatura: a tabela é a mesa, o array é a calculadora, e reescalonar já é conta de modelo. Quando o dado vier de um `DataFrame` — o que é o caso normal —, ele atravessa a [fronteira](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) antes: `rescale(df[colunas].to_numpy())`.

> **🔷 Conceito**
>
> Como sempre, é preciso usar julgamento. Se você pegasse um conjunto de dados grande e o filtrasse para conter só pessoas com altura entre 69,5 e 70,5 polegadas, é bem provável que a variação restante naquela dimensão seja só ruído — e talvez você não quisesse colocar o desvio padrão dela em pé de igualdade com o das outras dimensões. Reescalonar não é neutro: é uma decisão sobre o que conta como sinal.

> **💡 Dica — Na prática: `scikit-learn`**
>
> `scale` e `rescale` juntos são exatamente o `StandardScaler` do `scikit-learn`:
>
> ```python
> from sklearn.preprocessing import StandardScaler
>
> escalonador = StandardScaler()
> dados_reescalonados = escalonador.fit_transform(dados)
> ```
>
> `fit` calcula média e desvio padrão de cada coluna (o que você fez em `scale`, com `axis=0`); `transform` aplica a fórmula `(x - média) / desvio` (o que você fez em `rescale`, por *broadcasting*); `fit_transform` faz as duas coisas em sequência. Uma diferença que não está na interface: o `StandardScaler` divide por $n$, não por $n - 1$ — é o `ddof=0` do `numpy`, não o `ddof=1` do seu `scale` —, então os números não batem no último dígito, embora com centenas de linhas a diferença seja desprezível. A separação entre `fit` e `transform` importa na prática: você ajusta o escalonador **só** nos dados de treino, e aplica a mesma transformação — as mesmas médias e desvios, calculados no treino — aos dados de teste. Se você recalculasse média e desvio em cima do conjunto de teste, estaria vazando informação dele para dentro do seu pipeline antes mesmo de avaliar o modelo.
>
> O `scikit-learn` também tem `MinMaxScaler`, que reescalona para um intervalo fixo (geralmente `[0, 1]`) em vez de média 0 e desvio padrão 1 — útil quando a distribuição dos dados está longe de normal e desvio padrão não é uma unidade natural para eles.

## Um Parêntese: tqdm

> **📌 Nota**
>
> Esta seção corresponde a *An Aside: tqdm*, do capítulo 10 de Grus (2019).

Boa parte do que vem daqui para frente neste livro envolve cálculos que demoram — laços com dezenas de milhares de iterações, gradiente descendente rodando por centenas de passos. Quando um cálculo desses está em andamento, é bom saber que ele está de fato progredindo, e ter alguma ideia de quanto falta.

Uma forma de fazer isso é a biblioteca `tqdm`, que gera barras de progresso configuráveis.

Há só duas coisas que você realmente precisa saber sobre ela. A primeira é que envolver um iterável em `tqdm.tqdm` produz uma barra de progresso:

In [ ]:
import tqdm
import numpy as np

rng = np.random.default_rng(0)

for i in tqdm.tqdm(range(100)):
    # faz algo lento: sorteia um milhão de números e os ordena
    _ = np.sort(rng.random(1_000_000))

O `rng = np.random.default_rng(0)` não muda nada do que a barra mostra — nada aqui depende dos valores sorteados —, mas todo sorteio deste livro tem semente, sem exceção para exemplos de brinquedo.

O que aparece no terminal, ao vivo, é uma linha que se atualiza a cada iteração — algo como:

```
 57%|█████▋    | 57/100 [00:02<00:01, 26.43it/s]
```

Em particular, ela mostra que fração do laço já terminou, há quanto tempo está rodando, e quanto tempo ainda deve levar (a não ser que você esteja envolvendo um gerador, caso em que `tqdm` não tem como saber o tamanho total). Como o laço acima percorre um `range`, `tqdm.tqdm` sabe o total de antemão e consegue estimar o tempo restante.

> **🟩 Exemplo**
>
> `tqdm` escreve a barra no fluxo de erro padrão (`stderr`), não na saída padrão, e reescreve sempre a mesma linha. É por isso que a barra se atualiza no lugar em vez de encher a tela — e também por que ela não vai junto quando você redireciona a saída padrão do programa para um arquivo: a barra continua no terminal, separada do resultado.

A segunda coisa é que dá para configurar a descrição da barra enquanto ela roda. Para isso, você precisa capturar o iterador do `tqdm` com um `with`. No caso de estarmos apenas envolvendo `range`, dá para usar diretamente `tqdm.trange`:

In [ ]:
from typing import List

def primes_up_to(n: int) -> List[int]:
    primes = [2]

    with tqdm.trange(3, n) as t:
        for i in t:
            # i é primo se nenhum primo menor o divide
            i_is_prime = not any(i % p == 0 for p in primes)
            if i_is_prime:
                primes.append(i)

            t.set_description(f"{len(primes)} primos")

    return primes

my_primes = primes_up_to(100_000)

len(my_primes)

Isso acrescenta uma descrição como a seguinte, com um contador que se atualiza conforme novos primos são encontrados:

```
5088 primos:  50%|████████               | 49529/99997 [00:03<00:03, 15905.90it/s]
```

> **🟩 Exemplo**
>
> `primes_up_to` não é o algoritmo mais rápido possível para achar primos — para cada `i`, ele testa divisibilidade contra **todos** os primos já encontrados, não só os menores que $\sqrt{i}$. Isso é proposital: o ponto aqui não é otimizar a busca por primos, é ter um laço genuinamente lento para demonstrar `tqdm` em cima de algo real, em vez de um `time.sleep` artificial.

Usar `tqdm` vai, de vez em quando, deixar seu código um pouco instável — às vezes a tela redesenha mal, às vezes o laço trava de verdade. E se você acidentalmente aninhar um laço `tqdm` dentro de outro laço `tqdm`, coisas estranhas podem acontecer. Ainda assim, os benefícios costumam superar essas desvantagens, então vamos usá-lo sempre que tivermos cálculos que demoram — a começar pela próxima seção, onde `tqdm.trange` acompanha os passos do gradiente descendente ajustando a primeira componente principal.

> **💡 Dica — Na prática: o que se faz com isso**
>
> `tqdm` não é algo que uma biblioteca de modelagem substitui — ele é ferramenta de operação, não de cálculo, e continua sendo a escolha usada em produção para acompanhar laços Python comuns. O que muda fora deste livro é a integração: `tqdm.pandas()` registra uma barra de progresso para `DataFrame.progress_apply`, e frameworks de treinamento de rede neural (o assunto dos capítulos [15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) e [16](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap16/index.html) deste livro) costumam ter uma barra de progresso embutida, no mesmo espírito, para acompanhar epoch a epoch de treino.

## Redução de Dimensionalidade

> **📌 Nota**
>
> Esta seção corresponde a *Dimensionality Reduction*, do capítulo 10 de Grus (2019).

Às vezes as dimensões "de verdade" (ou úteis) de um conjunto de dados não correspondem às dimensões que ele tem. Considere o conjunto de dados a seguir:

In [ ]:
import numpy as np

pca_data = np.array([
[20.9666776351559,-13.1138080189357],
[22.7719907680008,-19.8890894944696],
[25.6687103160153,-11.9956004517219],
[18.0019794950564,-18.1989191165133],
[21.3967402102156,-10.8893126308196],
[0.443696899177716,-19.7221132386308],
[29.9198322142127,-14.0958668502427],
[19.0805843080126,-13.7888747608312],
[16.4685063521314,-11.2612927034291],
[21.4597664701884,-12.4740034586705],
[3.87655283720532,-17.575162461771],
[34.5713920556787,-10.705185165378],
[13.3732115747722,-16.7270274494424],
[20.7281704141919,-8.81165591556553],
[24.839851437942,-12.1240962157419],
[20.3019544741252,-12.8725060780898],
[21.9021426929599,-17.3225432396452],
[23.2285885715486,-12.2676568419045],
[28.5749111681851,-13.2616470619453],
[29.2957424128701,-14.6299928678996],
[15.2495527798625,-18.4649714274207],
[26.5567257400476,-9.19794350561966],
[30.1934232346361,-12.6272709845971],
[36.8267446011057,-7.25409849336718],
[32.157416823084,-10.4729534347553],
[5.85964365291694,-22.6573731626132],
[25.7426190674693,-14.8055803854566],
[16.237602636139,-16.5920595763719],
[14.7408608850568,-20.0537715298403],
[6.85907008242544,-18.3965586884781],
[26.5918329233128,-8.92664811750842],
[-11.2216019958228,-27.0519081982856],
[8.93593745011035,-20.8261235122575],
[24.4481258671796,-18.0324012215159],
[2.82048515404903,-22.4208457598703],
[30.8803004755948,-11.455358009593],
[15.4586738236098,-11.1242825084309],
[28.5332537090494,-14.7898744423126],
[40.4830293441052,-2.41946428697183],
[15.7563759125684,-13.5771266003795],
[19.3635588851727,-20.6224770470434],
[13.4212840786467,-19.0238227375766],
[7.77570680426702,-16.6385739839089],
[21.4865983854408,-15.290799330002],
[12.6392705930724,-23.6433305964301],
[12.4746151388128,-17.9720169566614],
[23.4572410437998,-14.602080545086],
[13.6878189833565,-18.9687408182414],
[15.4077465943441,-14.5352487124086],
[20.3356581548895,-10.0883159703702],
[20.7093833689359,-12.6939091236766],
[11.1032293684441,-14.1383848928755],
[17.5048321498308,-9.2338593361801],
[16.3303688220188,-15.1054735529158],
[26.6929062710726,-13.306030567991],
[34.4985678099711,-9.86199941278607],
[39.1374291499406,-10.5621430853401],
[21.9088956482146,-9.95198845621849],
[22.2367457578087,-17.2200123442707],
[10.0032784145577,-19.3557700653426],
[14.045833906665,-15.871937521131],
[15.5640911917607,-18.3396956121887],
[24.4771926581586,-14.8715313479137],
[26.533415556629,-14.693883922494],
[12.8722580202544,-21.2750596021509],
[24.4768291376862,-15.9592080959207],
[18.2230748567433,-14.6541444069985],
[4.1902148367447,-20.6144032528762],
[12.4332594022086,-16.6079789231489],
[20.5483758651873,-18.8512560786321],
[17.8180560451358,-12.5451990696752],
[11.0071081078049,-20.3938092335862],
[8.30560561422449,-22.9503944138682],
[33.9857852657284,-4.8371294974382],
[17.4376502239652,-14.5095976075022],
[29.0379635148943,-14.8461553663227],
[29.1344666599319,-7.70862921632672],
[32.9730697624544,-15.5839178785654],
[13.4211493998212,-20.150199857584],
[11.380538260355,-12.8619410359766],
[28.672631499186,-8.51866271785711],
[16.4296061111902,-23.3326051279759],
[25.7168371582585,-13.8899296143829],
[13.3185154732595,-17.8959160024249],
[3.60832478605376,-25.4023343597712],
[39.5445949652652,-11.466377647931],
[25.1693484426101,-12.2752652925707],
[25.2884257196471,-7.06710309184533],
[6.77665715793125,-22.3947299635571],
[20.1844223778907,-16.0427471125407],
[25.5506805272535,-9.33856532270204],
[25.1495682602477,-7.17350567090738],
[15.6978431006492,-17.5979197162642],
[37.42780451491,-10.843637288504],
[22.974620174842,-10.6171162611686],
[34.6327117468934,-9.26182440487384],
[34.7042513789061,-6.9630753351114],
[15.6563953929008,-17.2196961218915],
[25.2049825789225,-14.1592086208169]
])
pca_data.shape

Grande parte da variação dos dados parece estar ao longo de uma única dimensão, que não corresponde nem ao eixo $x$ nem ao eixo $y$:

In [ ]:
# Figura: Dados com os eixos \"errados\
from matplotlib import pyplot as plt

plt.scatter(pca_data[:, 0], pca_data[:, 1], s=15, color="darkblue")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados com os eixos \"errados\"")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

Quando isso acontece, dá para usar uma técnica chamada **análise de componentes principais** (PCA, na sigla em inglês) para extrair uma ou mais dimensões que capturem o máximo possível da variação nos dados.

> **❗ Importante**
>
> Na prática, você não usaria essa técnica num conjunto de dados tão baixo-dimensional quanto este. Redução de dimensionalidade é útil principalmente quando o conjunto tem um número grande de dimensões e você quer achar um pequeno subconjunto delas que capture a maior parte da variação. Infelizmente, esse caso é difícil de ilustrar num formato de livro bidimensional — por isso o exemplo aqui tem só duas dimensões, mesmo a técnica sendo pensada para muitas.

> **📌 Nota**
>
> Esta seção depende do gradiente descendente do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) — especificamente de `gradient_step`, a função construída lá que treina praticamente todo modelo deste livro. Aqui ela vem de `scratch_np.gradient_descent`, em arrays: `v + step_size * gradient`, uma linha no lugar do `add(v, scalar_multiply(step_size, gradient))` do Capítulo 5, com a mesma convenção de sinal. Ela não ajusta parâmetro de modelo nenhum; ajusta a **direção** que melhor explica a variação dos dados. Se `gradient_step` não estiver familiar, vale revisitar aquele capítulo antes de continuar.

### Centralizando os dados

Como primeiro passo, precisamos transladar os dados para que cada dimensão tenha média 0:

In [ ]:
import numpy as np

def de_mean(X: np.ndarray) -> np.ndarray:
    """Recentra os dados para que cada coluna tenha média 0"""
    return X - X.mean(axis=0)

de_meaned = de_mean(pca_data)
assert np.allclose(de_meaned.mean(axis=0), 0)

É a linha do *broadcasting* da [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html): a média de cada coluna, forma `(2,)`, subtraída de cada uma das 99 linhas. O `assert` confere com `np.allclose`, não com `== 0` — a média de floats recentrados fica na casa de 10⁻¹⁵, não em zero exato.

(Se não fizermos isso, é bem provável que nossas técnicas identifiquem a própria média em vez da variação nos dados.)

Centralizar é a metade da [7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html) que sobrou: mesma translação, sem dividir pelo desvio padrão. Por que só a metade — e quando a outra também precisa vir junto — é o que a "Na prática" desta seção fecha.

In [ ]:
# Figura: Dados depois de remover a média
plt.scatter(de_meaned[:, 0], de_meaned[:, 1], s=15, color="darkblue")
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados depois de remover a média")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

A forma da nuvem de pontos não mudou nada — só o centro dela, que agora é a origem.

### A direção de maior variância

Agora, dada uma matriz $X$ de dados centralizados, podemos perguntar qual é a direção que captura a maior variância nos dados.

Por que variância, e não alguma outra medida de espalhamento? Porque projetar os dados numa única direção e depois tentar reconstruí-los a partir só dessa projeção perde exatamente a variação que ficou de fora dela — e a direção que **maximiza** a variância capturada é a mesma que **minimiza** o quanto se perde ao descartar o resto. São o mesmo problema, visto de dois lados; resolvemos pelo lado que dá uma função mais fácil de diferenciar.

> **🔷 Conceito**
>
> A direção de maior variância é a direção em que os dados menos se parecem com um ponto só — o que sobra de informação depois de projetar tudo nela e aceitar não guardar mais nada. Reduzir dimensionalidade é escolher, de propósito, o que descartar primeiro: o que varia menos.

Vale fixar também os dois sentidos da palavra **componente**, porque o restante desta seção usa os dois: primeiro, uma **componente principal** é uma *direção* — um vetor de magnitude 1, como o `fpc` que `first_principal_component` devolve daqui a pouco. Depois, quando um ponto é projetado nessa direção, o número que sai da projeção — o `v @ w` que a seção "Projetando e removendo", adiante, vai calcular — também é chamado de "a componente" daquele ponto. A primeira é uma seta no espaço original; a segunda é uma coordenada nova, medida ao longo dela.

Especificamente, dada uma direção $d$ (um vetor de magnitude 1), a projeção de cada linha $x$ da matriz sobre $d$ tem comprimento `x @ d` — é o quanto daquele ponto "cabe" naquela direção — e `X @ d` calcula os $n$ comprimentos de uma vez, um por linha. E todo vetor $w$ não nulo determina uma direção, se o reescalonarmos para ter magnitude 1:

In [ ]:
def direction(w: np.ndarray) -> np.ndarray:
    return w / np.linalg.norm(w)

Portanto, dado um vetor $w$ não nulo, podemos calcular a variância do nosso conjunto de dados na direção determinada por $w$:

In [ ]:
def directional_variance(X: np.ndarray, w: np.ndarray) -> float:
    """Devolve a variância de X na direção de w"""
    w_dir = direction(w)
    return np.sum((X @ w_dir) ** 2)

`X @ w_dir` é o vetor de comprimentos de projeção, um por ponto; elevar ao quadrado e somar é a soma dos quadrados de todos eles.

> **📌 Nota**
>
> Apesar do nome e do *docstring*, `directional_variance` não divide por $n$: é uma **soma de quadrados**, não a variância no sentido em que o resto deste livro usa a palavra. Como os dados já passaram por `de_mean`, essa soma é proporcional à variância de verdade — dividir por $n$ mudaria a escala do número, mas não a direção $w$ que o maximiza, que é tudo que interessa aqui. Se você for comparar este valor contra a fórmula de variância que já conhece, é este o motivo do descompasso.

Gostaríamos de achar a direção que maximiza essa variância. Podemos fazer isso com gradiente descendente, assim que tivermos a função de gradiente:

In [ ]:
def directional_variance_gradient(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    """O gradiente da variância direcional em relação a w"""
    w_dir = direction(w)
    return 2 * (X @ w_dir) @ X

> **📌 Nota**
>
> Essa fórmula não é derivada aqui — vale aceitá-la como dada, mas com a razão de ela ser legítima: `directional_variance` soma termos $(v \cdot w_{dir})^2$, e a derivada de um quadrado como esse em relação a cada coordenada de $w$ é o dobro do produto interno vezes a coordenada correspondente de $v$, somado sobre todos os pontos. É o que `2 * (X @ w_dir) @ X` faz: `X @ w_dir` são os $n$ produtos internos; multiplicá-los por `X` pela esquerda pesa cada linha pelo seu produto e soma as linhas — a coordenada $i$ do resultado é $\sum_v 2\,(v \cdot w_{dir})\, v_i$. É cálculo de uma variável aplicado termo a termo, não uma fórmula nova.

E agora a primeira componente principal é justamente a direção que maximiza a função `directional_variance`:

In [ ]:
import tqdm
from scratch_np.gradient_descent import gradient_step

def first_principal_component(X: np.ndarray,
                              n: int = 100,
                              step_size: float = 0.1) -> np.ndarray:
    # começa com um palpite aleatório
    guess = np.ones(X.shape[1])

    with tqdm.trange(n) as t:
        for _ in t:
            dv = directional_variance(X, guess)
            gradient = directional_variance_gradient(X, guess)
            guess = gradient_step(guess, gradient, step_size)
            t.set_description(f"dv: {dv:.3f}")

    return direction(guess)

fpc = first_principal_component(de_meaned)
assert 0.923 < fpc[0] < 0.925
assert 0.382 < fpc[1] < 0.384
fpc

> **❗ Importante**
>
> Repare no sinal: `step_size` aqui é **positivo**. Todo o resto deste livro passa passo negativo para `gradient_step` — para andar *contra* o gradiente e minimizar algum erro, como o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/03-usando-o-gradiente.html) explica ao apresentar essa mesma função. Aqui não há erro para minimizar: queremos **maximizar** a variância direcional, então andamos **com** o gradiente. É o único uso ascendente do livro inteiro — e é o mesmo `gradient_step` do Capítulo 5, agora em arrays, usado ao contrário.

> **📌 Nota**
>
> O comentário diz "palpite aleatório", mas repare no código: `guess` começa sempre em `np.ones(X.shape[1])` — aqui, `[1.0, 1.0]`. Não há aleatoriedade nenhuma aqui — o gradiente parte sempre do mesmo lugar e segue sempre o mesmo caminho, então o resultado é determinístico e os dois `assert` acima valem em qualquer execução, sem precisar de `default_rng`. Uma inicialização genuinamente aleatória — como a que o [Capítulo 15](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html) usa para os pesos de uma rede neural — exigiria a semente para valer o mesmo.

No conjunto de dados centralizado, isso devolve a direção aproximadamente $(0{,}924,\ 0{,}383)$, que de fato parece capturar o eixo principal ao longo do qual nossos dados variam:

In [ ]:
# Figura: Primeira componente principal
escala = 30
plt.scatter(de_meaned[:, 0], de_meaned[:, 1], s=15, color="darkblue")
plt.plot([-escala * fpc[0], escala * fpc[0]],
        [-escala * fpc[1], escala * fpc[1]],
        color="black", linewidth=1.5)
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Primeira componente principal")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

### Projetando e removendo

Uma vez achada a direção que é a primeira componente principal, podemos projetar nossos dados nela para achar os valores dessa componente:

In [ ]:
def project(v: np.ndarray, w: np.ndarray) -> np.ndarray:
    """devolve a projeção de v na direção w"""
    projection_length = v @ w
    return projection_length * w

Se quisermos achar componentes adicionais, primeiro removemos as projeções dos dados:

In [ ]:
def remove_projection_from_vector(v: np.ndarray, w: np.ndarray) -> np.ndarray:
    """projeta v em w e subtrai o resultado de v"""
    return v - project(v, w)

def remove_projection(X: np.ndarray, w: np.ndarray) -> np.ndarray:
    """remove de cada linha de X a projeção dela em w"""
    return X - np.outer(X @ w, w)

residual = remove_projection(de_meaned, fpc)
assert np.allclose(residual @ fpc, 0)

`remove_projection_from_vector` é a versão de um ponto; `remove_projection` faz o mesmo para todas as linhas sem laço. `np.outer(a, w)` é o **produto externo**: a matriz cuja linha $i$ é `a[i] * w` — aqui, a projeção de cada linha de `X` em `w`, empilhadas. Subtraí-la de `X` é aplicar `remove_projection_from_vector` linha a linha de uma vez. O `assert` confere o que a figura seguinte mostra: não sobrou nada na direção de `fpc`.

Como este conjunto de exemplo é só bidimensional, depois de removermos a primeira componente o que sobra é efetivamente unidimensional:

In [ ]:
# Figura: Dados depois de remover a primeira componente principal
plt.scatter(residual[:, 0], residual[:, 1], s=15, color="darkblue")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados depois de remover a primeira componente principal")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

Os pontos ainda têm duas coordenadas, mas em duas dimensões, remover a projeção sobre uma direção deixa exatamente a direção ortogonal — daí a reta perfeita: toda a variação que sobrou está numa única direção, a segunda componente principal.

### Muitas componentes

Num conjunto de dados de maior dimensão, podemos achar quantas componentes quisermos iterando o processo: ache a direção de maior variância, remova a projeção nela, repita no que sobrou.

In [ ]:
def pca(X: np.ndarray, num_components: int) -> np.ndarray:
    """Devolve as componentes como linhas de uma matriz (num_components, d)"""
    components = []
    for _ in range(num_components):
        component = first_principal_component(X)
        components.append(component)
        X = remove_projection(X, component)

    return np.array(components)

O laço fica: é o algoritmo — uma componente por vez, removendo a projeção antes de procurar a próxima. A única mudança é o retorno: uma matriz `(k, d)`, uma componente por linha, para que `transform` seja um produto.

E então podemos **transformar** nossos dados para o espaço de dimensão menor gerado pelas componentes:

In [ ]:
def transform_vector(v: np.ndarray, components: np.ndarray) -> np.ndarray:
    return components @ v

def transform(X: np.ndarray, components: np.ndarray) -> np.ndarray:
    return X @ components.T

componentes = pca(de_meaned, 2)
transformado = transform(de_meaned, componentes)
componentes, transformado[:5]

`transform_vector` é `components @ v`: um produto interno por componente, os $k$ de uma vez. `transform` é o mesmo para todas as linhas, `X @ components.T` — forma `(n, d)` vezes `(d, k)`, que dá `(n, k)`: cada ponto reexpresso em $k$ coordenadas novas. A segunda componente saiu $(-0{,}383,\ 0{,}924)$, perpendicular à primeira — é a direção que a figura do resíduo prometia.

Com as duas componentes deste exemplo bidimensional, `transform` só reexpressa cada ponto num novo par de eixos — não reduz nada. O ganho aparece quando `num_components` é bem menor que o número de dimensões originais: um conjunto de centenas de colunas pode virar um punhado de componentes que ainda capturam a maior parte da variação. É o problema que ficou em aberto na [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html): a matriz de dispersão é uma ferramenta de triagem que não escala além de um punhado de dimensões. PCA é a resposta para quando há dimensões demais até para desenhar essa matriz.

Essa técnica é valiosa por duas razões. Primeiro, ela pode ajudar a limpar os dados, eliminando dimensões de ruído e consolidando dimensões fortemente correlacionadas. Segundo, depois de extrair uma representação de baixa dimensão dos dados, dá para usar uma variedade de técnicas que não funcionam tão bem em dados de alta dimensão — o assunto da [seção sobre a maldição da dimensionalidade](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-a-maldicao-da-dimensionalidade.html). Ela também é a resposta que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html), sobre seleção de atributos, prometia: uma forma de **remover** atributos em vez de criá-los, quando o número deles é grande demais.

Ao mesmo tempo, embora essa técnica possa ajudar a construir modelos melhores, ela também pode tornar esses modelos mais difíceis de interpretar. É fácil entender uma conclusão como "cada ano a mais de experiência aumenta o salário médio em 10 mil reais". É bem mais difícil dar sentido a "cada aumento de 0,1 na terceira componente principal aumenta o salário médio em 10 mil reais".

É aqui que se fecha a distância anunciada na abertura deste capítulo. O dado que chegou pelos canos do [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) — torto, incompleto, cheio de outliers e datas impossíveis — passou por sete seções e chega a este ponto explorado, tipado, limpo, agregado, em escala comum e, agora, com dimensões a menos. É exatamente o formato que a máquina de gradiente descendente do [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) espera receber.

> **💡 Dica — Na prática: `scikit-learn`**
>
> ```python
> from sklearn.decomposition import PCA
>
> modelo = PCA(n_components=2)
> modelo.fit(dados)
> dados_transformados = modelo.transform(dados)
>
> modelo.explained_variance_ratio_  # quanto cada componente explica
> ```
>
> A diferença mais importante não é de interface, é de método. O que você construiu acima acha uma componente de cada vez, por gradiente descendente, e remove a projeção antes de achar a próxima — um processo iterativo e aproximado. O `scikit-learn` resolve para **todas** as componentes de uma vez, por decomposição em valores singulares (SVD), o que é ao mesmo tempo mais rápido e numericamente mais estável — sem depender de tamanho de passo, número de iterações ou um palpite inicial, como o seu `first_principal_component` depende. Trocar listas por arrays não mudou o método: `X @ w_dir` só faz de uma vez o que o laço fazia ponto a ponto; a subida do gradiente, com seus 100 passos e seu palpite inicial, é a mesma. O SVD é outro algoritmo.
>
> Duas pegadinhas que `PCA` do `scikit-learn` não avisa sozinho: primeiro, `fit` centraliza os dados automaticamente (o equivalente ao seu `de_mean`), mas **não** reescalona por desvio padrão — se as dimensões têm unidades muito diferentes, é preciso rodar `StandardScaler` (a [seção 7.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/06-reescalonamento.html)) antes de `PCA`, senão a dimensão de maior variância numérica domina as componentes só por causa da escala, não porque carregue mais informação de verdade. Segundo, o sinal de cada componente é **matematicamente** arbitrário: $(0{,}924,\ 0{,}383)$ e $(-0{,}924,\ -0{,}383)$ descrevem a mesma direção, e nenhuma das duas está "mais certa" que a outra — a sua implementação poderia devolver qualquer uma, dependendo do palpite inicial. O `scikit-learn`, porém, **não** deixa isso ao acaso: ele passa o resultado por `svd_flip`, que fixa o sinal por convenção. Rodando os quatro solvers de `PCA` sobre estes mesmos dados, todos devolvem $(0{,}924,\ 0{,}383)$, sempre.
>
> Isso é útil e é exatamente o tipo de decisão que uma biblioteca toma por você sem avisar. Útil, porque duas execuções concordam e o seu gráfico não vira de cabeça para baixo entre uma rodada e outra. Silencioso, porque em lugar nenhum da saída está escrito que houve uma escolha — e se você comparar as suas componentes com as de outra ferramenta, que adote a convenção oposta, os sinais vão discordar sem que nada esteja errado.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 10 de Grus (2019) sugere:

- [pandas](https://pandas.pydata.org/) deixou de ser recomendação e passou a ser ferramenta: o [Capítulo 6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html) o apresentou para ler dado, e este capítulo o usa para explorar, limpar, filtrar, agrupar e agregar. O que este capítulo escreveu à mão — `bucketize`, `try_parse_row`, `day_over_day_changes`, o laço do `max` por símbolo — foi escrito uma vez, para mostrar o mecanismo, e substituído na linha seguinte pela chamada que se usa de verdade. O que **não** foi substituído é o que fica do lado do modelo — `scale`, `rescale` e a PCA da [seção 7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html) — mais `correlation_matrix`, que sobrevive ao lado do `df.corr()` porque a armadilha do `rowvar` é a lição. *Python for Data Analysis* (O'Reilly), de Wes McKinney — o criador do `pandas` —, é a referência para aprendê-lo a sério, e o [guia de usuário oficial](https://pandas.pydata.org/docs/user_guide/index.html) cobre bem os capítulos de `groupby` e de série temporal, que são os dois que este livro só encosta.
- O `scikit-learn` tem uma [família inteira de funções de decomposição de matriz](https://scikit-learn.org/stable/modules/decomposition.html), incluindo PCA — mas resolvida por decomposição em valores singulares, não pelo gradiente descendente que você usou na [seção 7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html). Para o tratamento formal de por que isso funciona, veja G{\'e}ron (2022).

## Referências

- **G{\'e}ron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed.. O'Reilly Media. 2022.
- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.